# Ink Detection from 3D X-ray Scans

This notebook detects ink in 3D x-ray scans of ancient papyrus fragments using a 3D CNN model.
Supports both CPU (small dataset) and GPU (full dataset) execution.

## 1. Configuration & Imports

In [2]:
# Device Configuration - CHANGE THIS FOR CPU/GPU
USE_GPU = False  # Set to True for GPU (NVIDIA)
%pip install tensorflow
import os
if not USE_GPU:
    os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # Disable GPU
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Tuple, Optional
from PIL import Image
import gc

# Verify device
device_type = "GPU" if tf.config.list_physical_devices('GPU') and USE_GPU else "CPU"
print(f"Running on: {device_type}")
print(f"TensorFlow version: {tf.__version__}")

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement tensorflow (from versions: none)

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for tensorflow


ModuleNotFoundError: No module named 'tensorflow'

## 2. Utility Functions (RLE Encoding/Decoding & Data Loading)

In [ ]:
def numeric_slice_key(path: Path):
    """Sort TIFF files numerically"""
    return int(path.stem) if path.stem.isdigit() else path.name

def first_tiff_path(patch_dir: Path) -> Path:
    """Get first TIFF slice in surface_volume folder"""
    tiffs = sorted((patch_dir / "surface_volume").glob("*.tif"), key=numeric_slice_key)
    if not tiffs:
        raise FileNotFoundError(f"No TIFF slices found in {patch_dir / 'surface_volume'}")
    return tiffs[0]

def rle_encode(mask: np.ndarray) -> str:
    """Encode binary mask to RLE format (Kaggle-style)"""
    pixels = mask.astype(np.uint8).ravel()
    pixels = np.concatenate([[0], pixels, [0]])
    changes = np.where(pixels[1:] != pixels[:-1])[0] + 1
    changes[1::2] -= changes[::2]
    return " ".join(str(x) for x in changes)

def rle_decode(rle: object, shape: Tuple[int, int]) -> np.ndarray:
    """Decode RLE format to binary mask (1-indexed Kaggle-style)"""
    flat = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    if rle is None or (isinstance(rle, float) and np.isnan(rle)):
        return flat.reshape(shape)
    
    rle = str(rle).strip()
    if not rle:
        return flat.reshape(shape)
    
    values = np.asarray([int(v) for v in rle.split()], dtype=np.int64)
    starts = values[0::2] - 1
    lengths = values[1::2]
    ends = starts + lengths
    for start, end in zip(starts, ends):
        flat[start:end] = 1
    return flat.reshape(shape)

In [ ]:
def load_surface_volume(patch_dir: Path, num_slices: int = 65, downsample: int = 1) -> np.ndarray:
    """
    Load all TIFF slices from surface_volume folder.
    
    Args:
        patch_dir: Path to patch folder
        num_slices: Number of slices to load (max 65)
        downsample: Downsampling factor to reduce memory (1=no downsample)
    
    Returns:
        3D numpy array (height, width, slices)
    """
    surface_dir = patch_dir / "surface_volume"
    tiffs = sorted(surface_dir.glob("*.tif"), key=numeric_slice_key)[:num_slices]
    
    slices = []
    for tiff_path in tiffs:
        img = np.asarray(Image.open(tiff_path))
        if downsample > 1:
            img = img[::downsample, ::downsample]
        slices.append(img)
    
    volume = np.stack(slices, axis=-1).astype(np.float32)
    # Clear memory
    del slices
    gc.collect()
    return volume

def load_training_sample(patch_dir: Path, num_slices: int = 65, downsample: int = 1) -> Tuple[np.ndarray, np.ndarray]:
    """
    Load training sample with label.
    Returns: (3D volume, 2D ink mask)
    """
    volume = load_surface_volume(patch_dir, num_slices=num_slices, downsample=downsample)
    label_path = patch_dir / "inklabels.png"
    label = np.asarray(Image.open(label_path)) > 0
    
    if downsample > 1:
        label = label[::downsample, ::downsample]
    
    return volume, label.astype(np.float32)

def load_test_sample(patch_dir: Path, num_slices: int = 65, downsample: int = 1) -> np.ndarray:
    """
    Load test sample (no label).
    Returns: 3D volume
    """
    return load_surface_volume(patch_dir, num_slices=num_slices, downsample=downsample)

## 3. Data Loading & Preparation

In [ ]:
# Set paths
TRAIN_DIR = Path("/home/raffaele/Documenti/GitHub/ELIVA/Image_Segmentation/train")
TEST_DIR = Path("/home/raffaele/Documenti/GitHub/ELIVA/Image_Segmentation/test")

# Get all patch directories
train_patches = sorted([p for p in TRAIN_DIR.iterdir() if p.is_dir()])
test_patches = sorted([p for p in TEST_DIR.iterdir() if p.is_dir()])

print(f"Training samples: {len(train_patches)}")
print(f"Test samples: {len(test_patches)}")

# ========== CONFIGURATION: CHANGE FOR FULL DATASET ==========
# CPU (test mode - minimal resources)
NUM_TRAIN_SAMPLES = 2      # Reduced from 3
DOWNSAMPLE = 8             # Increased from 4 (256x256 instead of 512x512)
NUM_SLICES = 32            # Reduced from 65 (use subset of z-slices)

# GPU (full mode - uncomment and use instead)
# NUM_TRAIN_SAMPLES = len(train_patches)
# DOWNSAMPLE = 1
# NUM_SLICES = 65
# ===========================================================

print(f"\nUsing {NUM_TRAIN_SAMPLES} training samples")
print(f"Downsampling factor: {DOWNSAMPLE} (image size: {2048//DOWNSAMPLE}x{2048//DOWNSAMPLE})")
print(f"Number of slices per volume: {NUM_SLICES}/65")

Training samples: 42
Test samples: 16

Using 2 training samples
Downsampling factor: 8 (image size: 256x256)
Number of slices per volume: 32/65


In [ ]:
# Load training data
print("Loading training data...")
X_train = []
y_train = []

for i, patch_dir in enumerate(train_patches[:NUM_TRAIN_SAMPLES]):
    print(f"Loading {i+1}/{NUM_TRAIN_SAMPLES}: {patch_dir.name}", end="")
    try:
        volume, label = load_training_sample(patch_dir, num_slices=NUM_SLICES, downsample=DOWNSAMPLE)
        X_train.append(volume)
        y_train.append(label)
        print(f" ✓ Shape: {volume.shape}")
        gc.collect()  # Clear memory after each sample
    except Exception as e:
        print(f" ✗ Error: {e}")

print(f"\nSuccessfully loaded {len(X_train)} training samples")

Loading training data...
Loading 1/2: p_0s2t5eg93al2

 ✓ Shape: (256, 256, 32)
Loading 2/2: p_11502d5ne53i ✓ Shape: (256, 256, 32)

Successfully loaded 2 training samples


In [ ]:
# Normalize and reshape for training
# Normalize volumes to [0, 1]
X_train = [x / (np.max(x) + 1e-6) for x in X_train]

# Add channel dimension: (H, W, D) -> (H, W, D, 1)
X_train_normalized = np.array([x[..., np.newaxis] for x in X_train])
y_train_normalized = np.array([y[..., np.newaxis] for y in y_train])

print(f"Training data shape: {X_train_normalized.shape}")
print(f"Training labels shape: {y_train_normalized.shape}")
print(f"Data type: {X_train_normalized.dtype}")
print(f"Value range (X): [{X_train_normalized.min():.3f}, {X_train_normalized.max():.3f}]")
print(f"Value range (y): [{y_train_normalized.min():.1f}, {y_train_normalized.max():.1f}]")

Training data shape: (2, 256, 256, 32, 1)
Training labels shape: (2, 256, 256, 1)
Data type: float32
Value range (X): [0.000, 1.000]
Value range (y): [0.0, 1.0]


## 4. Build 3D CNN Model (with one layer removed)

In [ ]:
def build_model_3d_cnn(input_shape: Tuple[int, ...]) -> keras.Model:
    """
    Build a 3D CNN model for ink detection.
    Architecture: Conv3D -> Conv3D -> MaxPool3D -> Conv3D -> UpSampling3D -> Collapse3Dto2D -> Conv2D (output)
    Note: One convolutional layer is removed compared to standard architectures.
    """
    model = keras.Sequential([
        # Input block
        layers.Input(shape=input_shape),
        
        # Encoder block 1: Conv3D -> Conv3D
        layers.Conv3D(32, kernel_size=3, activation='relu', padding='same'),
        layers.Conv3D(32, kernel_size=3, activation='relu', padding='same'),
        layers.MaxPooling3D(pool_size=2),  # Reduce spatial dimensions
        
        # Encoder block 2: Conv3D (ONE LAYER REMOVED - baseline had Conv3D->Conv3D)
        layers.Conv3D(64, kernel_size=3, activation='relu', padding='same'),
        layers.MaxPooling3D(pool_size=2),
        
        # Decoder block: UpSampling + Conv3D
        layers.UpSampling3D(size=2),
        layers.Conv3D(32, kernel_size=3, activation='relu', padding='same'),
        layers.UpSampling3D(size=2),
        
        # Collapse depth dimension to 2D by averaging across slices
        layers.Lambda(lambda x: tf.reduce_mean(x, axis=3, keepdims=False)),
        
        # Output layer: binary mask (now 2D: H, W, C) - use Conv2D
        layers.Conv2D(1, kernel_size=3, activation='sigmoid', padding='same'),
    ])
    
    return model

# Build model
input_shape = X_train_normalized.shape[1:]  # (H, W, D, C)
model = build_model_3d_cnn(input_shape)

# Compile
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Model Architecture:")
model.summary()

Model Architecture:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv3d (Conv3D)                 │ (None, 256, 256, 32,   │           896 │
│                                 │ 32)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_1 (Conv3D)               │ (None, 256, 256, 32,   │        27,680 │
│                                 │ 32)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d (MaxPooling3D)    │ (None, 128, 128, 16,   │             0 │
│                                 │ 32)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_2 (Conv3D)               │ (None, 128, 128, 16,   │        55,360 │
│                                 │ 64)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_1 (MaxPooling3D)  │ (None, 64, 64, 8, 64)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling3d (UpSampling3D)    │ (None, 128, 128, 16,   │             0 │
│                                 │ 64)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_3 (Conv3D)               │ (None, 128, 128, 16,   │        55,328 │
│                                 │ 32)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling3d_1 (UpSampling3D)  │ (None, 256, 256, 32,   │             0 │
│                                 │ 32)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 256, 256, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 256, 256, 1)    │           289 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 139,553 (545.13 KB)

 Trainable params: 139,553 (545.13 KB)

 Non-trainable params: 0 (0.00 B)

## 5. Train Model

In [ ]:
# Training configuration
EPOCHS = 5 if not USE_GPU else 50     # Reduced from 10 for CPU testing
BATCH_SIZE = 1 if not USE_GPU else 4  # Smaller batch for CPU

print(f"Training configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Device: {device_type}")

# Train with memory cleanup
history = model.fit(
    X_train_normalized,
    y_train_normalized,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    verbose=1
)

# Clear memory
gc.collect()

Training configuration:
  Epochs: 5
  Batch size: 1
  Device: CPU
Epoch 1/5


W0000 00:00:1778233804.430158   40793 cpu_allocator_impl.cc:82] Allocation of 268435456 exceeds 10% of free system memory.
W0000 00:00:1778233804.693472   40793 cpu_allocator_impl.cc:82] Allocation of 268435456 exceeds 10% of free system memory.
W0000 00:00:1778233805.582279   40793 cpu_allocator_impl.cc:82] Allocation of 67108864 exceeds 10% of free system memory.
W0000 00:00:1778233805.792570   40793 cpu_allocator_impl.cc:82] Allocation of 67108864 exceeds 10% of free system memory.
W0000 00:00:1778233805.999669   40793 cpu_allocator_impl.cc:82] Allocation of 67108864 exceeds 10% of free system memory.


In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Model Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['accuracy'], label='Train Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Model Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 6. Prediction on Test Set

In [ ]:
# Load test data
print("Loading test data...")
X_test = []
test_patch_names = []

for i, patch_dir in enumerate(test_patches):
    print(f"Loading {i+1}/{len(test_patches)}: {patch_dir.name}", end="")
    try:
        volume = load_test_sample(patch_dir, num_slices=NUM_SLICES, downsample=DOWNSAMPLE)
        X_test.append(volume)
        test_patch_names.append(patch_dir.name)
        print(f" ✓")
        gc.collect()  # Clear memory after each sample
    except Exception as e:
        print(f" ✗ Error: {e}")

print(f"\nSuccessfully loaded {len(X_test)} test samples")

In [ ]:
# Normalize test data
X_test = [x / (np.max(x) + 1e-6) for x in X_test]
X_test_normalized = np.array([x[..., np.newaxis] for x in X_test])

print(f"Test data shape: {X_test_normalized.shape}")

In [ ]:
# Make predictions
print("Making predictions on test set...")
predictions = model.predict(X_test_normalized, verbose=1)

# Apply threshold to get binary masks
threshold = 0.5
binary_predictions = (predictions > threshold).astype(np.uint8)

print(f"Predictions shape: {predictions.shape}")
print(f"Binary predictions shape: {binary_predictions.shape}")

## 7. Generate Submission File (RLE Format)

In [ ]:
# Prepare submission
submission_data = []

for i, (patch_name, pred) in enumerate(zip(test_patch_names, binary_predictions)):
    # pred shape: (H, W, D, 1), we need to squeeze and collapse to 2D
    pred_2d = pred.squeeze()  # Remove last dimension
    
    # Collapse 3D mask to 2D by taking max across depth dimension
    if len(pred_2d.shape) == 3:
        pred_2d = np.max(pred_2d, axis=-1)
    
    # Encode to RLE
    rle_encoded = rle_encode(pred_2d)
    submission_data.append({"Id": patch_name, "Predicted": rle_encoded})
    
    if i < 3:
        print(f"{patch_name}: RLE length = {len(rle_encoded)}")

# Create DataFrame
submission_df = pd.DataFrame(submission_data)
print(f"\nSubmission DataFrame:")
print(submission_df.head())

In [ ]:
# Save submission file
output_path = Path("/home/raffaele/Documenti/GitHub/ELIVA/Image_Segmentation/submission.csv")
submission_df.to_csv(output_path, index=False)
print(f"Submission file saved to: {output_path}")
print(f"File size: {output_path.stat().st_size / 1024:.2f} KB")

## 8. Visualization (Optional)

In [ ]:
# Visualize predictions on first test sample
if len(predictions) > 0:
    sample_idx = 0
    
    # Get first slice of volume and prediction
    volume_slice = X_test_normalized[sample_idx, :, :, 0, 0]
    pred_slice = binary_predictions[sample_idx, :, :, 0, 0]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    axes[0].imshow(volume_slice, cmap='gray')
    axes[0].set_title(f"Surface Volume Slice - {test_patch_names[sample_idx]}")
    axes[0].axis('off')
    
    axes[1].imshow(pred_slice, cmap='gray')
    axes[1].set_title("Predicted Ink Mask")
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()

## Summary

✓ Model trained and evaluated  
✓ Predictions generated on test set  
✓ Submission file saved (RLE format)  

### Next Steps for Full Training:
1. Set `USE_GPU = True` and `CUDA_VISIBLE_DEVICES` appropriately
2. Set `NUM_TRAIN_SAMPLES = len(train_patches)` to use all training data
3. Set `DOWNSAMPLE = 1` for full resolution
4. Increase `EPOCHS` to 50+ for better convergence
5. Set `BATCH_SIZE = 4` or higher based on GPU memory